# 04 — Comparación BGE y TF-IDF

Este notebook resume M5. No vuelve a entrenar los modelos: lee el resultado reproducible generado en Khipu.

La pregunta es simple: **¿BGE mejora la decisión de triaje frente a TF-IDF cuando ambos usan las mismas filas?**

## 1. Cargar el resultado de M5

BGE representa el significado de una narrativa mediante una lista de números llamada **embedding**. TF-IDF representa la importancia de palabras y pares de palabras.

In [1]:
import json
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

report_path = PROJECT_ROOT / 'reports/modeling/bge_sample_results.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print(f'Reporte: {report_path.relative_to(PROJECT_ROOT)}')
print(f"Modelo BGE: {report['bge']['model']}")
print(f"Revisión fija: {report['bge']['resolved_revision']}")

Reporte: reports\modeling\bge_sample_results.json
Modelo BGE: BAAI/bge-large-en-v1.5
Revisión fija: d4aa6901d3a41ba39fb536a557fa166f842b0e09


## 2. Muestra temporal usada

Los dos métodos se comparan sobre las mismas filas. La columna **sin texto compartido** cuenta las narrativas que no aparecieron en periodos anteriores usados para aprender. Esta es la vista principal porque reduce el beneficio de memorizar plantillas.

In [2]:
period_names = {
    'fit': 'Ajuste',
    'calibration': 'Calibración',
    'validation': 'Validación',
}
sample_rows = []
for split, values in report['sample'].items():
    sample_rows.append({
        'Periodo': period_names[split],
        'Filas': values['rows'],
        'Sin texto compartido': values['no_shared_text_rows'],
    })

sample_table = pd.DataFrame(sample_rows)
display(sample_table)

,Periodo,Filas,Sin texto compartido
0,Ajuste,120000,120000
1,Calibración,40000,28633
2,Validación,80000,64955


## 3. Comparación principal

- **Macro-F1 en T1:** calcula F1 por motivo y da el mismo peso a cada motivo. Un valor mayor es mejor.
- **Precisión promedio en T2–T4:** resume qué tan bien ordena el modelo los casos positivos cuando son poco frecuentes. Un valor mayor es mejor.

La columna `Cambio BGE` es el resultado de BGE menos el de TF-IDF. Un número positivo favorece BGE.

In [3]:
target_names = {
    'T1': 'Motivo CFPB',
    'T2': 'Algún relief registrado',
    'T3': 'Relief monetario registrado',
    'T4': 'Respuesta no oportuna CFPB',
}
metric_names = {
    'macro_f1': 'Macro-F1',
    'average_precision': 'Precisión promedio',
}
comparison_rows = []
for target, values in report['comparison'].items():
    comparison_rows.append({
        'Objetivo': f"{target} — {target_names[target]}",
        'Métrica': metric_names[values['primary_metric']],
        'TF-IDF': values['tfidf'],
        'BGE': values['bge'],
        'Cambio BGE': values['bge_improvement'],
    })

comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table.round(4))

,Objetivo,Métrica,TF-IDF,BGE,Cambio BGE
0,T1 — Motivo CFPB,Macro-F1,0.1968,0.2367,0.0399
1,T2 — Algún relief registrado,Precisión promedio,0.5662,0.5666,0.0004
2,T3 — Relief monetario registrado,Precisión promedio,0.2640,0.2576,-0.0064
3,T4 — Respuesta no oportuna CFPB,Precisión promedio,0.0791,0.0624,-0.0167


## 4. Decisión por objetivo

BGE mejora claramente T1. En T2 la diferencia es menor a 0.001, por lo que no compensa usar un método más costoso. En T3 y T4 BGE queda por debajo.

No necesitamos forzar un único modelo para todo el triaje. Elegimos la alternativa más útil y simple para cada objetivo.

In [4]:
decisions = pd.DataFrame([
    {'Objetivo': 'T1', 'Modelo conservado': 'BGE + producto', 'Razón': 'Mejora Macro-F1 en 0.0399'},
    {'Objetivo': 'T2', 'Modelo conservado': 'TF-IDF + producto', 'Razón': 'El empate favorece el método más simple'},
    {'Objetivo': 'T3', 'Modelo conservado': 'TF-IDF + producto', 'Razón': 'Supera a BGE'},
    {'Objetivo': 'T4', 'Modelo conservado': 'Regla por producto', 'Razón': 'Sigue superando a los modelos de texto'},
])
display(decisions)

,Objetivo,Modelo conservado,Razón
0,T1,BGE + producto,Mejora Macro-F1 en 0.0399
1,T2,TF-IDF + producto,El empate favorece el método más simple
2,T3,TF-IDF + producto,Supera a BGE
3,T4,Regla por producto,Sigue superando a los modelos de texto


## 5. Artefacto reproducible

DVC guarda los embeddings y clasificadores grandes fuera de Git. El archivo `.dvc` registra exactamente qué versión del artefacto corresponde a este resultado.

In [5]:
dvc_path = PROJECT_ROOT / 'artifacts/models/bge_sample.dvc'
dvc_pointer = yaml.safe_load(dvc_path.read_text(encoding='utf-8'))['outs'][0]
artifact_summary = pd.Series({
    'Ruta DVC': 'artifacts/models/bge_sample',
    'Hash': dvc_pointer['md5'],
    'Tamaño en bytes': dvc_pointer['size'],
    'Archivos': dvc_pointer['nfiles'],
    'Iteraciones máximas permitidas': report['bge']['linear_max_iter'],
})
display(artifact_summary.to_frame('Valor'))

,Valor
Ruta DVC,artifacts/models/bge_sample
Hash,009e3b35e25d9df095cf753e0a05f041.dir
Tamaño en bytes,1090649311
Archivos,15
Iteraciones máximas permitidas,120


## 6. Qué sigue

Los ocho runs de M5 están publicados en [MLflow](https://dagshub.com/WinzCode/capstone-claims-triage.mlflow/#/experiments/0).

BGE continúa hacia M6 porque sus embeddings permiten buscar reclamos con significado parecido y crear grupos semánticos. Esa utilidad es diferente de predecir T1–T4: BGE puede aportar buenos vecinos aunque no mejore todos los objetivos supervisados.